# Unit 13 — Digital Electronics Solutions

Worked answers for the short-answer exercises, and reference programs for the programming exercises.
Each programming solution is also saved as a runnable program in the `assets` folder (for example `assets/ex10.py`); run it with `python assets/ex10.py < input.txt`.
Each short-answer answer is checked by a cell that runs a small circuit evaluator kept in `assets/verify/circuit_eval.py`.
Simplify answers are also checked with Unit 8's Boolean evaluator, `bool_eval.py`, which finds every minimal sum of products.

## Exercise 1

Work down the netlist with `A = 1`, `B = 0`, `C = 1`:

| Gate | Inputs | Output |
|---|---|---|
| `p = NOR(A, C)` | 1, 1 | NOT (1 OR 1) = 0 |
| `q = XNOR(B, p)` | 0, 0 | the inputs are equal, so 1 |
| `out = NAND(q, C)` | 1, 1 | NOT (1 AND 1) = 0 |

**Answer:** `0`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C
p = NOR(A, C)
q = XNOR(B, p)
out = NAND(q, C)"""

assert str(circuit_eval.evaluate(netlist, {"A": 1, "B": 0, "C": 1})) == "0"

## Exercise 2

`p` is NOT A, `q` is NAND of `p` and B, and the output is `q` XOR A.

| A | B | p = NOT A | q = NAND(p, B) | out = XOR(q, A) |
|---|---|---|---|---|
| 0 | 0 | 1 | NOT (1 AND 0) = 1 | 1 ⊕ 0 = 1 |
| 0 | 1 | 1 | NOT (1 AND 1) = 0 | 0 ⊕ 0 = 0 |
| 1 | 0 | 0 | NOT (0 AND 0) = 1 | 1 ⊕ 1 = 0 |
| 1 | 1 | 0 | NOT (0 AND 1) = 1 | 1 ⊕ 1 = 0 |

Read the last column from top to bottom.
(As an expression, `q` is `~(~A * B)`, which is `A + ~B` by De Morgan's law; the output is 1 only when A is 0 and `q` is 1, that is, only in row `(0,0)`.)

**Answer:** `1000`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B
p = NOT(A)
q = NAND(p, B)
out = XOR(q, A)"""

column = "".join(str(circuit_eval.evaluate(netlist, {"A": a, "B": b})) for a, b in product((0, 1), repeat=2))
assert str(column) == "1000"

## Exercise 3

The output is `(A + B) * ~(B * C)`: it is FALSE when `p` is 0 (A = B = 0) or when `q` is 0 (B = C = 1).

| A | B | C | p = OR(A, B) | q = NAND(B, C) | out = AND(p, q) |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 1 | **0** |
| 0 | 0 | 1 | 0 | 1 | **0** |
| 0 | 1 | 0 | 1 | 1 | 1 |
| 0 | 1 | 1 | 1 | 0 | **0** |
| 1 | 0 | 0 | 1 | 1 | 1 |
| 1 | 0 | 1 | 1 | 1 | 1 |
| 1 | 1 | 0 | 1 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 | **0** |

The four FALSE rows, in counting-up order:

**Answer:** `(0,0,0), (0,0,1), (0,1,1), (1,1,1)`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C
p = OR(A, B)
q = NAND(B, C)
out = AND(p, q)"""

assert str(circuit_eval.solutions(netlist, 0)) == "(0,0,0), (0,0,1), (0,1,1), (1,1,1)"

## Exercise 4

The output is `(A ⊕ C) + ~(A + B)`.

| A | B | C | p = XOR(A, C) | q = NOR(A, B) | out = OR(p, q) |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 1 | 1 |
| 0 | 0 | 1 | 1 | 1 | 1 |
| 0 | 1 | 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 | 0 | 1 |
| 1 | 0 | 1 | 0 | 0 | 0 |
| 1 | 1 | 0 | 1 | 0 | 1 |
| 1 | 1 | 1 | 0 | 0 | 0 |

Five rows have output 1.
(Check: `p` is 1 in the four rows where A and C differ, and `q` adds one more row, `(0,0,0)`; its other row, `(0,0,1)`, already has `p` = 1.)

**Answer:** `5`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C
p = XOR(A, C)
q = NOR(A, B)
out = OR(p, q)"""

assert str(circuit_eval.count(netlist)) == "5"

## Exercise 5

The circuit is `~(A + B) + ~(B * C)`.
Simplify it with the laws from Unit 8:

- De Morgan: `~(A + B)` is `~A * ~B`, and `~(B * C)` is `~B + ~C`, so the circuit is `~A * ~B + ~B + ~C`.
- Absorption: `~A * ~B + ~B` is `~B`, so the circuit is `~B + ~C`. It is 0 only when B = C = 1.

Now each option:

- **(A)** `~(A + B) + ~(B * C)` is the circuit read gate by gate. **Yes.**
- **(B)** `~A * ~B + ~B + ~C` is the circuit after De Morgan. **Yes.**
- **(C)** `~(A + B + B * C)`: absorption makes the inside `A + B`, so this is `~A * ~B`. At `(1,0,0)` it gives 0, but the circuit gives 1. **No.**
- **(D)** `~A + ~B + ~C`: at `(0,1,1)` it gives 1, but the circuit gives 0. **No.**
- **(E)** `~B + ~C` is the simplified circuit. **Yes.**

**Answer:** `A, B, E`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C
p = NOR(A, B)
q = NAND(B, C)
out = OR(p, q)"""
options = {"A": "~(A + B) + ~(B * C)", "B": "~A * ~B + ~B + ~C", "C": "~(A + B + B * C)", "D": "~A + ~B + ~C", "E": "~B + ~C"}
rows = [dict(zip("ABC", bits)) for bits in product((0, 1), repeat=3)]
chosen = [label for label in sorted(options) if all(bool_eval.evaluate(options[label], row) == circuit_eval.evaluate(netlist, row) for row in rows)]
assert str(", ".join(chosen) or "NONE") == "A, B, E"

## Exercise 6

Read the sketch one gate at a time:

- A and B go into an AND gate, whose output is labelled `p`.
- C goes into a NOT gate, whose output is labelled `q`.
- `p` and `q` go into a NOR gate, whose output is labelled `r`.
- A (the long top wire) and `r` go into an XOR gate, which gives `out`.

So the circuit is `p = AND(A, B)`, `q = NOT(C)`, `r = NOR(p, q)`, `out = XOR(r, A)`.
Now each option:

- **(A)** `r` is an OR gate, not a NOR. **No** (its output is the opposite of the circuit's in every row).
- **(B)** The same gates with the same wires. Only the order of the two inputs of AND, NOR and XOR is swapped, and these gates give the same output whichever way round their inputs go. **Yes.**
- **(C)** B and C are swapped: `p` uses C and the NOT gate takes B. At `(0,1,0)` the circuit gives `0` but (C) gives `1`. **No.**
- **(D)** The last gate is an XNOR, not an XOR. **No** (opposite output in every row).
- **(E)** Exactly the sketch. **Yes.**

The check below compares each option's truth table with (E), the netlist read straight from the sketch.

**Answer:** `B, E`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

sketch = """INPUTS A B C
p = AND(A, B)
q = NOT(C)
r = NOR(p, q)
out = XOR(r, A)"""
options = {
    "A": """INPUTS A B C
p = AND(A, B)
q = NOT(C)
r = OR(p, q)
out = XOR(r, A)""",
    "B": """INPUTS A B C
p = AND(B, A)
q = NOT(C)
r = NOR(q, p)
out = XOR(A, r)""",
    "C": """INPUTS A B C
p = AND(A, C)
q = NOT(B)
r = NOR(p, q)
out = XOR(r, A)""",
    "D": """INPUTS A B C
p = AND(A, B)
q = NOT(C)
r = NOR(p, q)
out = XNOR(r, A)""",
    "E": """INPUTS A B C
p = AND(A, B)
q = NOT(C)
r = NOR(p, q)
out = XOR(r, A)""",
}
rows = [dict(zip("ABC", bits)) for bits in product((0, 1), repeat=3)]
chosen = [label for label in sorted(options) if all(circuit_eval.evaluate(options[label], row) == circuit_eval.evaluate(sketch, row) for row in rows)]
assert str(", ".join(chosen) or "NONE") == "B, E"

## Exercise 7

Gate by gate: `p` is `~(A * C)` and `q` is `B + C`, so the circuit is `~(A * C) * (B + C)`.

Simplify:

- De Morgan: `~(A * C)` is `~A + ~C`, so the circuit is `(~A + ~C) * (B + C)`.
- Distribute: `~A * B + ~A * C + B * ~C + C * ~C`.
- `C * ~C` is 0, so this is `~A * B + ~A * C + B * ~C`.
- Consensus: `~A * B` is covered by the other two terms (if A = 0 and B = 1, then either C = 1, so `~A * C` is 1, or C = 0, so `B * ~C` is 1), so drop it.

Check with the truth table: the circuit is 1 at `(0,0,1)`, `(0,1,0)`, `(0,1,1)` and `(1,1,0)`.
`~A * C` covers `(0,0,1)` and `(0,1,1)`; `B * ~C` covers `(0,1,0)` and `(1,1,0)`.
No single term covers all four rows, and each of these two terms is the only one that covers `(0,0,1)` or `(1,1,0)`, so this two-term answer is the only minimal one.

**Answer:** `~A * C + B * ~C`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C
p = NAND(A, C)
q = OR(B, C)
out = AND(p, q)"""

minimal = bool_eval.minimal_sops(circuit_eval.to_expression(netlist))
assert len(bool_eval.minimal_sops(circuit_eval.to_expression(netlist))) == 1
assert str(minimal[0]) == "~A * C + B * ~C"

## Exercise 8

First find `p` and `q` in each row, then ask which gate turns them into the wanted output:

| A | B | p = AND(A, B) | q = NOR(A, B) | wanted out |
|---|---|---|---|---|
| 0 | 0 | 0 | 1 | 1 |
| 0 | 1 | 0 | 0 | 0 |
| 1 | 0 | 0 | 0 | 0 |
| 1 | 1 | 1 | 0 | 1 |

`p` and `q` are never both 1, so the gate only ever sees the input pairs `(0,1)`, `(0,0)` and `(1,0)`, and it must give 1, 0 and 1 on them:

- **(A)** AND gives 0 on `(0,1)`. **No.**
- **(B)** NAND gives 1 on `(0,0)`. **No.**
- **(C)** OR gives 1, 0, 1. **Yes.**
- **(D)** NOR gives 0 on `(0,1)`. **No.**
- **(E)** XOR gives 1, 0, 1. **Yes** (it differs from OR only on `(1,1)`, which never happens here).
- **(F)** XNOR gives 0 on `(0,1)`. **No.**

**Answer:** `C, E`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B
p = AND(A, B)
q = NOR(A, B)
out = GATE(p, q)"""
options = {"A": "AND", "B": "NAND", "C": "OR", "D": "NOR", "E": "XOR", "F": "XNOR"}
chosen = []
for label in sorted(options):
    filled = netlist.replace("GATE", options[label])
    column = "".join(str(circuit_eval.evaluate(filled, {"A": a, "B": b})) for a, b in product((0, 1), repeat=2))
    if column == "1001":
        chosen.append(label)
assert str(", ".join(chosen) or "NONE") == "C, E"

## Exercise 9

No gate uses C, but C is still on the `INPUTS` line, so every row counts twice: once with C = 0 and once with C = 1.

Work out the output for each `(A,B)`:

| A | B | p = XNOR(A, B) | out = NOR(p, B) |
|---|---|---|---|
| 0 | 0 | 1 | 0 |
| 0 | 1 | 0 | 0 |
| 1 | 0 | 0 | 1 |
| 1 | 1 | 1 | 0 |

Only `(A,B) = (1,0)` gives 1, so the TRUE triples are `(1,0,0)` and `(1,0,1)`.
(As an expression: `~((A ⊙ B) + B)` is `~(A ⊙ B) * ~B`, which is `A * ~B`.)

**Answer:** `2`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C
p = XNOR(A, B)
out = NOR(p, B)"""

assert str(circuit_eval.count(netlist)) == "2"

## Exercise 10

**Notice:** one function, `gate`, holds all eight truth tables.
NAND, NOR and XNOR are worked out as AND, OR and XOR and then flipped with `1 - answer`, so there are only four cases to get right.
With 0 and 1, AND is the product `x * y` and OR is `max(x, y)`; XOR is 1 exactly when the two values differ.
A one-value line (`NOT 0`) leaves `y` at 0, which `gate` never looks at for `BUFFER` and `NOT`.

In [ ]:
def gate(name, x, y):
    if name == "BUFFER":
        answer = x
    elif name == "NOT":
        answer = 1 - x
    elif name == "AND" or name == "NAND":
        answer = x * y
    elif name == "OR" or name == "NOR":
        answer = max(x, y)
    else:
        answer = int(x != y)
    if name == "NAND" or name == "NOR" or name == "XNOR":
        answer = 1 - answer
    return answer


n = int(input())
for i in range(n):
    parts = input().split()
    x = int(parts[1])
    y = 0
    if len(parts) == 3:
        y = int(parts[2])
    print(gate(parts[0], x, y))

## Exercise 11

**Notice:** `words` cuts a line at spaces, `=`, brackets and commas, so `q = NOR(p, C)` becomes `["q", "NOR", "p", "C"]`: the gate's name, its type, then its one or two inputs.
The dict `value` starts with the circuit inputs (from the `INPUTS` line and the last line) and gains each gate's output as it is worked out, so a later gate can use an earlier gate's name just like an input.
The line of values is read only after the g gate lines, because it comes last in the input.

In [ ]:
def gate(name, x, y):
    if name == "BUFFER":
        answer = x
    elif name == "NOT":
        answer = 1 - x
    elif name == "AND" or name == "NAND":
        answer = x * y
    elif name == "OR" or name == "NOR":
        answer = max(x, y)
    else:
        answer = int(x != y)
    if name == "NAND" or name == "NOR" or name == "XNOR":
        answer = 1 - answer
    return answer


def words(line):
    result = []
    word = ""
    for ch in line + " ":
        if ch in " =(),":
            if word != "":
                result.append(word)
            word = ""
        else:
            word = word + ch
    return result


g = int(input())
names = words(input())
lines = []
for i in range(g):
    lines.append(input())
bits = input().split()

value = {}
for i in range(1, len(names)):
    value[names[i]] = int(bits[i - 1])
for line in lines:
    w = words(line)
    x = value[w[2]]
    y = 0
    if len(w) == 4:
        y = value[w[3]]
    value[w[0]] = gate(w[1], x, y)
    print(w[0] + " " + str(value[w[0]]))

## Exercise 12

**Notice:** the same simulator, run once for each row.
Row number `row` (0, 1, 2, …) is turned into input values by taking its binary digits from the right: the last input gets `row % 2`, the one before it the next digit, and so on, so the rows come out in counting-up order.
An input that no gate uses still doubles the number of rows, because the count of rows comes from the `INPUTS` line, not from the gates.

In [ ]:
def gate(name, x, y):
    if name == "BUFFER":
        answer = x
    elif name == "NOT":
        answer = 1 - x
    elif name == "AND" or name == "NAND":
        answer = x * y
    elif name == "OR" or name == "NOR":
        answer = max(x, y)
    else:
        answer = int(x != y)
    if name == "NAND" or name == "NOR" or name == "XNOR":
        answer = 1 - answer
    return answer


def words(line):
    result = []
    word = ""
    for ch in line + " ":
        if ch in " =(),":
            if word != "":
                result.append(word)
            word = ""
        else:
            word = word + ch
    return result


def run(lines, value):
    last = ""
    for line in lines:
        w = words(line)
        x = value[w[2]]
        y = 0
        if len(w) == 4:
            y = value[w[3]]
        value[w[0]] = gate(w[1], x, y)
        last = w[0]
    return value[last]


g = int(input())
names = words(input())
lines = []
for i in range(g):
    lines.append(input())

k = len(names) - 1
total = 1
for i in range(k):
    total = total * 2

column = ""
for row in range(total):
    value = {}
    rest = row
    for i in range(k, 0, -1):
        value[names[i]] = rest % 2
        rest = rest // 2
    column = column + str(run(lines, value))
print(column)

## Exercise 13

Write each gate as an expression:

- `p = ~(A * B)`
- `q = C ⊕ D`
- `r = ~(p + D) = ~p * ~D = A * B * ~D` (De Morgan, then `~~(A * B)` is `A * B`)
- `s = q ⊕ A = A ⊕ C ⊕ D`
- `out = r + s = A * B * ~D + (A ⊕ C ⊕ D)`

`A ⊕ C ⊕ D` is 1 exactly when an odd number of A, C, D are 1.
That happens for 4 of the 8 choices of A, C, D, and B is free, so `s` is 1 in 8 of the 16 rows.

`r` is 1 when A = 1, B = 1 and D = 0; that is the rows `(1,1,0,0)` and `(1,1,1,0)`.
In `(1,1,0,0)` A, C, D are 1, 0, 0 (odd), so `s` already counts it.
In `(1,1,1,0)` they are 1, 1, 0 (even), so this row is new.

The TRUE rows are `(0,0,0,1)`, `(0,0,1,0)`, `(0,1,0,1)`, `(0,1,1,0)`, `(1,0,0,0)`, `(1,0,1,1)`, `(1,1,0,0)`, `(1,1,1,0)` and `(1,1,1,1)`: 8 + 1 = 9.

**Answer:** `9`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C D
p = NAND(A, B)
q = XOR(C, D)
r = NOR(p, D)
s = XOR(q, A)
out = OR(r, s)"""

assert str(circuit_eval.count(netlist)) == "9"

## Exercise 14

An XNOR gate is 1 when its two inputs are equal, so `x ⊙ y` is `x ⊕ y` flipped.
Along the chain:

- `p = A ⊙ B` is 1 when A and B have an even number of 1s.
- `q = p ⊙ C`: each XNOR flips the answer of an XOR, and two flips cancel, so `q = A ⊕ B ⊕ C`, which is 1 when A, B, C have an odd number of 1s.
- `r = q ⊙ D` flips once more: `r` is 1 when A, B, C, D have an **even** number of 1s.

The output is `r * A`: A must be 1, and then A, B, C, D have an even number of 1s exactly when B, C, D have an **odd** number of 1s.
The triples `(B,C,D)` with an odd number of 1s are `(0,0,1)`, `(0,1,0)`, `(1,0,0)` and `(1,1,1)`.

Check one row, `(1,0,0,1)`: `p` = 1 ⊙ 0 = 0, `q` = 0 ⊙ 0 = 1, `r` = 1 ⊙ 1 = 1, `out` = 1 * 1 = 1.

**Answer:** `(1,0,0,1), (1,0,1,0), (1,1,0,0), (1,1,1,1)`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C D
p = XNOR(A, B)
q = XNOR(p, C)
r = XNOR(q, D)
out = AND(r, A)"""

assert str(circuit_eval.solutions(netlist)) == "(1,0,0,1), (1,0,1,0), (1,1,0,0), (1,1,1,1)"

## Exercise 15

Gate by gate, the circuit is `(~(A * B) + ~(C + D)) * (A ⊙ C)`.

`s = A ⊙ C` is 1 only when A = C, so split into the two cases:

- **A = 0 and C = 0.** Then `~(A * B)` is 1, so `r` is 1 and the output is 1, whatever B and D are. This gives the term `~A * ~C`.
- **A = 1 and C = 1.** Then `~(C + D)` is 0, so `r` is `~(1 * B)`, which is `~B`. The output is 1 when B = 0, whatever D is. This gives the term `A * ~B * C`.

So the circuit is `A * ~B * C + ~A * ~C`.
It is 1 on the rows `(0,0,0,0)`, `(0,0,0,1)`, `(0,1,0,0)`, `(0,1,0,1)`, `(1,0,1,0)` and `(1,0,1,1)`.

Can it be shorter?
The two groups of rows share no row, and no bigger term fits the second group: `~B * C` would also cover `(0,0,1,0)` and `A * C` would also cover `(1,1,1,0)`, where the circuit is 0.
So two terms with 2 + 3 letters is the only minimal sum of products.
The terms are written in Unit 8's order (the term whose first letter is plain `A` comes before the one starting `~A`).

**Answer:** `A * ~B * C + ~A * ~C`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C D
p = NAND(A, B)
q = NOR(C, D)
r = OR(p, q)
s = XNOR(A, C)
out = AND(r, s)"""

minimal = bool_eval.minimal_sops(circuit_eval.to_expression(netlist))
assert len(bool_eval.minimal_sops(circuit_eval.to_expression(netlist))) == 1
assert str(minimal[0]) == "A * ~B * C + ~A * ~C"

## Exercise 16

A NOR gate with the same wire on both inputs is a NOT gate: `~(x + x)` is `~x`.

- `p = NOR(A, A) = ~A`
- `q = NOR(B, C) = ~(B + C)`
- `r = NOR(p, q) = ~(~A + ~(B + C))`. By De Morgan this is `~~A * ~~(B + C)`, which is `A * (B + C)`.
- `out = NOR(r, r) = ~r = ~(A * (B + C))`

Simplify the output with De Morgan twice: `~(A * (B + C))` is `~A + ~(B + C)`, which is `~A + ~B * ~C`.

It is 0 only when A = 1 and B + C = 1, and each term is needed: `~A` alone misses `(1,0,0)`, and `~B * ~C` alone misses `(0,1,0)`.
Neither term can lose a letter: `~B` or `~C` alone would cover `(1,0,1)` or `(1,1,0)`, where the circuit is 0.

**Answer:** `~A + ~B * ~C`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C
p = NOR(A, A)
q = NOR(B, C)
r = NOR(p, q)
out = NOR(r, r)"""

minimal = bool_eval.minimal_sops(circuit_eval.to_expression(netlist))
assert len(bool_eval.minimal_sops(circuit_eval.to_expression(netlist))) == 1
assert str(minimal[0]) == "~A + ~B * ~C"

## Exercise 17

The circuit is `(A + B) * ~(C * (B ⊙ D))`.
It is FALSE when either part of the AND is 0:

- **`p` = 0:** A = 0 and B = 0, any C and D: `(0,0,0,0)`, `(0,0,0,1)`, `(0,0,1,0)`, `(0,0,1,1)`.
- **`r` = 0:** C = 1 and B = D (so `q` = 1):
  - B = D = 0: `(0,0,1,0)` (already listed) and `(1,0,1,0)`
  - B = D = 1: `(0,1,1,1)` and `(1,1,1,1)`

Put the seven rows in counting-up order.

Check one row, `(1,0,1,0)`: `p` = 1 + 0 = 1, `q` = 0 ⊙ 0 = 1, `r` = NAND(1, 1) = 0, `out` = 1 * 0 = 0.

**Answer:** `(0,0,0,0), (0,0,0,1), (0,0,1,0), (0,0,1,1), (0,1,1,1), (1,0,1,0), (1,1,1,1)`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
import circuit_eval

netlist = """INPUTS A B C D
p = OR(A, B)
q = XNOR(B, D)
r = NAND(C, q)
out = AND(p, r)"""

assert str(circuit_eval.solutions(netlist, 0)) == "(0,0,0,0), (0,0,0,1), (0,0,1,0), (0,0,1,1), (0,1,1,1), (1,0,1,0), (1,1,1,1)"

## Exercise 18

**Notice:** Exercise 12's loop over the rows, keeping only the rows whose output is the wanted value.
Each kept row is turned into text as it is found, with `", "` added before every tuple except the first, so the list never ends in a stray comma.
The count is printed first, but it is only known after the loop, so the tuples are built up in the string `answer` and printed at the end; when nothing matched, the second line is `NONE`.
The `INPUTS` line gives the tuple's columns and their order, so a one-input circuit prints `(0)` and `(1)`, and an unused input still has its own column.

In [ ]:
def gate(name, x, y):
    if name == "BUFFER":
        answer = x
    elif name == "NOT":
        answer = 1 - x
    elif name == "AND" or name == "NAND":
        answer = x * y
    elif name == "OR" or name == "NOR":
        answer = max(x, y)
    else:
        answer = int(x != y)
    if name == "NAND" or name == "NOR" or name == "XNOR":
        answer = 1 - answer
    return answer


def words(line):
    result = []
    word = ""
    for ch in line + " ":
        if ch in " =(),":
            if word != "":
                result.append(word)
            word = ""
        else:
            word = word + ch
    return result


def run(lines, value):
    last = ""
    for line in lines:
        w = words(line)
        x = value[w[2]]
        y = 0
        if len(w) == 4:
            y = value[w[3]]
        value[w[0]] = gate(w[1], x, y)
        last = w[0]
    return value[last]


g = int(input())
names = words(input())
lines = []
for i in range(g):
    lines.append(input())
want = int(input())

k = len(names) - 1
total = 1
for i in range(k):
    total = total * 2

found = 0
answer = ""
for row in range(total):
    value = {}
    rest = row
    for i in range(k, 0, -1):
        value[names[i]] = rest % 2
        rest = rest // 2
    if run(lines, value) == want:
        if found > 0:
            answer = answer + ", "
        text = "("
        for i in range(1, k + 1):
            if i > 1:
                text = text + ","
            text = text + str(value[names[i]])
        answer = answer + text + ")"
        found = found + 1
print(found)
if found == 0:
    print("NONE")
else:
    print(answer)

## Exercise 19

`A ⊙ B` is 1 when A = B: its result column is `1001`.
Work out each circuit:

- **(A)** `p` = `~(A * B)`. Then `q` = `~(A * p)` and `r` = `~(B * p)`, and `out` = `~(q * r)` = `~q + ~r` = `A * p + B * p` = `(A + B) * ~(A * B)`. That is 1 when exactly one input is 1: `A ⊕ B`, column `0110`. **No** (it is the famous four-NAND XOR).
- **(B)** The same first four gates give `s` = `A ⊕ B`, and `out = NAND(s, s)` = `~s`, which is `A ⊙ B`. **Yes.**
- **(C)** `p` = `~A`, `q` = `~B`, `r` = `~(~A * ~B)` = `A + B`, and `out = NAND(r, r)` = `~(A + B)`, column `1000`. **No** (a NOR).
- **(D)** `p` = `~A`, `q` = `~B`, `r` = `~(A * B)`, `s` = `~(~A * ~B)` = `A + B`, and `out` = `~(r * s)` = `~r + ~s` = `A * B + ~A * ~B`. That is 1 when both inputs are 1 or both are 0: `A ⊙ B`. **Yes.**

| A | B | (A) | (B) | (C) | (D) |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 1 | 1 | 1 |
| 0 | 1 | 1 | 0 | 0 | 0 |
| 1 | 0 | 1 | 0 | 0 | 0 |
| 1 | 1 | 0 | 1 | 0 | 1 |

**Answer:** `B, D`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

target = "A ⊙ B"
options = {
    "A": """INPUTS A B
p = NAND(A, B)
q = NAND(A, p)
r = NAND(B, p)
out = NAND(q, r)""",
    "B": """INPUTS A B
p = NAND(A, B)
q = NAND(A, p)
r = NAND(B, p)
s = NAND(q, r)
out = NAND(s, s)""",
    "C": """INPUTS A B
p = NAND(A, A)
q = NAND(B, B)
r = NAND(p, q)
out = NAND(r, r)""",
    "D": """INPUTS A B
p = NAND(A, A)
q = NAND(B, B)
r = NAND(A, B)
s = NAND(p, q)
out = NAND(r, s)""",
}
rows = [dict(zip("AB", bits)) for bits in product((0, 1), repeat=2)]
chosen = [label for label in sorted(options) if all(circuit_eval.evaluate(options[label], row) == bool_eval.evaluate(target, row) for row in rows)]
assert str(", ".join(chosen) or "NONE") == "B, D"

## Exercise 20

Turn the target into NANDs with De Morgan: `A * B + ~C` is `~(~(A * B) * ~~C)`, which is `~(~(A * B) * C)`, that is `NAND(NAND(A, B), C)`.
So a two-layer NAND circuit needs `NAND(A, B)` and then a NAND of that with **C itself** (not `~C`).

- **(A)** `q` = `~C` and `r` = `NAND(q, q)` = `~~C` = `C`, so `out` = `NAND(~(A * B), C)`. **Yes** (the two extra gates just give back C).
- **(B)** `q` = `~C`, so `out` = `~(~(A * B) * ~C)` = `A * B + C`. At `(0,0,0)` it gives 0, but the target gives 1. **No.**
- **(C)** `out` = `NAND(~(A * B), C)`, exactly the form above. **Yes.**
- **(D)** `q` = `A * B` and `r` = `~C`, so `out` = `~(A * B * ~C)` = `~A + ~B + C`. At `(0,0,0)` it gives 1 like the target, but at `(1,1,0)` it gives 0 where the target gives 1. **No.**

**Answer:** `A, C`

In [ ]:
from itertools import product
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

target = "A * B + ~C"
options = {
    "A": """INPUTS A B C
p = NAND(A, B)
q = NAND(C, C)
r = NAND(q, q)
out = NAND(p, r)""",
    "B": """INPUTS A B C
p = NAND(A, B)
q = NAND(C, C)
out = NAND(p, q)""",
    "C": """INPUTS A B C
p = NAND(A, B)
out = NAND(p, C)""",
    "D": """INPUTS A B C
p = NAND(A, B)
q = NAND(p, p)
r = NAND(C, C)
out = NAND(q, r)""",
}
rows = [dict(zip("ABC", bits)) for bits in product((0, 1), repeat=3)]
chosen = [label for label in sorted(options) if all(circuit_eval.evaluate(options[label], row) == bool_eval.evaluate(target, row) for row in rows)]
assert str(", ".join(chosen) or "NONE") == "A, C"

## Exercise 21

Work from the output back.
`out = NAND(t, D)` is `~t + ~D`, so the circuit is 1 whenever D = 0.
That leaves only the rows with **D = 1**:

- `r = NOR(B, D)` is `~(B + 1)` = 0.
- `s = XNOR(q, r)` = `q ⊙ 0`, which is `~q`. Since `q = NAND(p, C)`, `s` is `p * C` = `(A ⊕ B) * C`.
- `t = OR(s, A)` = `(A ⊕ B) * C + A`.
- `out` = `~t`: it needs A = 0, and then `A ⊕ B` is just B, so `t` = `B * C` and `out` = `~(B * C)` = `~B + ~C`.

So for D = 1 the output is `~A * ~B + ~A * ~C`, and for D = 0 it is 1:

`out = ~D + D * (~A * ~B + ~A * ~C)`, and since `~D + D * X` is `~D + X`, this is `~A * ~B + ~A * ~C + ~D`.

Each term is needed and none can lose a letter:

- `~D` is the only term that covers `(1,1,1,0)`.
- `~A * ~B` is the only term that covers `(0,0,1,1)`; `~A` alone would also cover `(0,1,1,1)`, where the circuit is 0.
- `~A * ~C` is the only term that covers `(0,1,0,1)`.

**Answer:** `~A * ~B + ~A * ~C + ~D`

In [ ]:
import sys

sys.path.insert(0, "assets/verify")
sys.path.insert(0, "../unit-08-boolean-algebra/assets/verify")
import bool_eval
import circuit_eval

netlist = """INPUTS A B C D
p = XOR(A, B)
q = NAND(p, C)
r = NOR(B, D)
s = XNOR(q, r)
t = OR(s, A)
out = NAND(t, D)"""

minimal = bool_eval.minimal_sops(circuit_eval.to_expression(netlist))
assert len(bool_eval.minimal_sops(circuit_eval.to_expression(netlist))) == 1
assert str(minimal[0]) == "~A * ~B + ~A * ~C + ~D"

## Exercise 22

**Notice:** both netlists run on the same row, one row at a time in counting-up order, and the loop stops at the first row where their outputs differ.
The second netlist's `INPUTS` line is read and thrown away (`input()` on its own), since the statement promises it matches the first.
Each netlist keeps its own list of gate lines, and a fresh `value` dict is made for every row, filled by the first run and then re-used by the second: any gate name the second netlist shares with the first is simply overwritten in order, and every name it reads was set earlier in its own netlist, so the sharing never mixes up the two circuits.

In [ ]:
def gate(name, x, y):
    if name == "BUFFER":
        answer = x
    elif name == "NOT":
        answer = 1 - x
    elif name == "AND" or name == "NAND":
        answer = x * y
    elif name == "OR" or name == "NOR":
        answer = max(x, y)
    else:
        answer = int(x != y)
    if name == "NAND" or name == "NOR" or name == "XNOR":
        answer = 1 - answer
    return answer


def words(line):
    result = []
    word = ""
    for ch in line + " ":
        if ch in " =(),":
            if word != "":
                result.append(word)
            word = ""
        else:
            word = word + ch
    return result


def run(lines, value):
    last = ""
    for line in lines:
        w = words(line)
        x = value[w[2]]
        y = 0
        if len(w) == 4:
            y = value[w[3]]
        value[w[0]] = gate(w[1], x, y)
        last = w[0]
    return value[last]


g = int(input())
names = words(input())
first = []
for i in range(g):
    first.append(input())
g = int(input())
input()
second = []
for i in range(g):
    second.append(input())

k = len(names) - 1
total = 1
for i in range(k):
    total = total * 2

answer = "EQUIVALENT"
row = 0
while row < total and answer == "EQUIVALENT":
    value = {}
    rest = row
    for i in range(k, 0, -1):
        value[names[i]] = rest % 2
        rest = rest // 2
    if run(first, value) != run(second, value):
        answer = "("
        for i in range(1, k + 1):
            if i > 1:
                answer = answer + ","
            answer = answer + str(value[names[i]])
        answer = answer + ")"
    row = row + 1
print(answer)